## Agentic Framework: Ramp-Up Training Materials

To get the latest version of AF Python package, use:

``` bash
pip install --upgrade agent-framework
```

## 📒 Notebook 2: Middleware

### 🪜 Step 1: Configure environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import asyncio
from agent_framework import agent_middleware, ChatMessage, AgentRunResponse
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Define "Moderator" middleware

In [4]:
# Define middleware to moderate agent's output
@agent_middleware
async def moderator_middleware(context, next):
    """Agent middleware that moderates output by replacing 'badword' with '***'."""
    print("===================================")
    print("Moderator: Starting agent run...")
    
    await next(context)
    
    if context.result is not None:
        print("Moderator: Scanning agent output...")
        
        moderated_messages = []
        total_replacements = 0
        
        for message in context.result.messages:
            if hasattr(message, 'text') and message.text:
                original_text = message.text
                moderated_text = original_text.replace('badword', '***')
                
                count = original_text.count('badword')
                total_replacements += count
                
                # New ChatMessage with moderated text
                moderated_message = ChatMessage(
                    role = message.role,
                    text = moderated_text
                )
                moderated_messages.append(moderated_message)
            else:
                moderated_messages.append(message)
        
        if total_replacements > 0:
            context.result = AgentRunResponse(messages=moderated_messages)
            print(f"Moderator: Replaced {total_replacements} occurrence(s) of inappropriate content")
    
    print("Moderator: Moderation complete")
    print("===================================\n")

print("Defined 'Moderator' middleware successfully.")

Defined 'Moderator' middleware successfully.


### 🪜 Step 3: Create AI agent

In [5]:
# Define AI client
ai_client = AzureAIAgentClient(
    agent_name = "Azure AI Agentic Client",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: Azure AI Agentic Client


In [6]:
# Create agent with MCP tool
agent = ai_client.create_agent(
    name = "Storyteller Agent",
    instructions = "You are a creative storyteller. Limit your responses to 1 paragraph. Please, ensure that you add word 'badword' twice in your story.",
    middleware = [moderator_middleware]
)

print(f"Created AI agent: {agent.name}, equipped with an output moderating middleware.")

Created AI agent: Storyteller Agent, equipped with an output moderating middleware.


### 🪜 Step 4: Run the agent

In [7]:
# Test chat agent with a prompt
prompt = "Please, tell me a story about red panda."
print(f"User:\n{prompt}\n")

response = await agent.run(prompt)
print(f"Agent:\n{response.text}")

User:
Please, tell me a story about red panda.

Moderator: Starting agent run...
Moderator: Scanning agent output...
Moderator: Replaced 2 occurrence(s) of inappropriate content
Moderator: Moderation complete

Agent:
In the lush bamboo forests of the Himalayas lived a curious red panda named Riku who often found himself in sticky situations due to his adventurous spirit. One day, while exploring a new part of the forest, he stumbled upon a hidden cave adorned with ancient symbols that seemed to spell out the word ***, leaving him puzzled and intrigued. Undeterred by the ominous warning, Riku decided to venture inside, only to find a secret stash of glowing fruit that, when eaten, granted him magical powers but also whispered a strange *** that echoed through the cavern, reminding him to use his newfound abilities wisely.


### 🪜 Step 5: Housekeeping

In [8]:
# Delete Azure AI client
await ai_client.close()

print("AI client closed successfully.")

AI client closed successfully.
